# 05 — 100 alternative 80:20 partitions (Sections 3.9, 4.5)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### EXACT V4 M2/M3 REPEATED 80:20 PARTITION-SENSITIVITY ANALYSIS

In [ ]:
# ======================================================================
# EXACT V4 M2/M3 REPEATED 80:20 PARTITION-SENSITIVITY ANALYSIS
# CORRECTED VERSION
# ======================================================================
#
# This script:
#
# 1. Loads the original 144-patient TCGA survival cohort.
# 2. Loads the original 3,992-feature Stage-C eNESH matrix.
# 3. Loads the corresponding calibrated expression matrix.
# 4. Reconstructs the original seed-42 115/29 split.
# 5. RESTORES patient order exactly as Stage D did.
# 6. Repeats development-only zero-variance filtering.
# 7. Verifies that the resulting 3,669-feature M2/M3 matrices match
#    the saved Stage-D matrices numerically.
# 8. Repeats V4 final 3-fold tuning for M2 and M3.
# 9. Requires seed-42 learner, K, hyperparameters and C-index to
#    reproduce the saved V4 analysis.
# 10. ONLY IF ALL AUDITS PASS, repeats the complete procedure across
#     100 alternative event-stratified 80:20 partitions.
#
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
)

from sksurv.linear_model import (
    CoxnetSurvivalAnalysis,
)

from sksurv.metrics import (
    concordance_index_censored,
)


warnings.filterwarnings("ignore")


# ======================================================================
# 1. GLOBAL SETTINGS
# ======================================================================

RANDOM_SEED = 42

INNER_FOLDS = 3

N_FINAL_SEEDS = 5

TOP_K_VALUES = [
    10,
    25,
    50,
]

ZERO_VARIANCE_THRESHOLD = 1e-10


# ----------------------------------------------------------------------
# FINAL sensitivity analysis
# ----------------------------------------------------------------------

N_SPLITS = 100


# ----------------------------------------------------------------------
# Original manuscript/internal-test values
# ----------------------------------------------------------------------

ORIGINAL_M2_C = 0.6433

ORIGINAL_M3_C_EXACT = 0.640127388535


# ======================================================================
# 2. PATHS
# ======================================================================

DOWNLOADS = (
    BASE
)


STAGE_D_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)


STAGE_C_REWIRING_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)


STAGE_C_EXPRESSION_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_144_CALIBRATED"
)


# ----------------------------------------------------------------------
# Stage-D files
# ----------------------------------------------------------------------

SPLIT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_LOCKED_SPLIT.csv"
)


COHORT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)


M2_SAVED_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)


M3_SAVED_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M3_REWIRING_RAW.csv"
)


# ----------------------------------------------------------------------
# Original Stage-C source matrices
# ----------------------------------------------------------------------

ENESH_FULL_FILE = (
    STAGE_C_REWIRING_DIR
    / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)


EXPRESSION_FULL_FILE = (
    STAGE_C_EXPRESSION_DIR
    / "STAGE_C_144_CALIBRATED_DEG_ANCHORS.csv"
)


# ----------------------------------------------------------------------
# V4 final saved specifications
# ----------------------------------------------------------------------

V4_DIR = (
    STAGE_D_DIR
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)


V4_SPECS_FILE = (
    V4_DIR
    / "STAGE_D_V4_OOF_FINAL_SPECS.json"
)


# ----------------------------------------------------------------------
# Revision output
# ----------------------------------------------------------------------

OUTPUT_DIR = (
    STAGE_D_DIR
    / "REVISION_EXACT_V4_SPLIT_SENSITIVITY"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ======================================================================
# 3. V4 LEARNER GRID
# ======================================================================

LEARNER_GRID = [

    # ---------------------------
    # Random Survival Forest
    # ---------------------------

    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },

    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
    },

    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": 0.30,
    },

    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": 0.30,
    },


    # ---------------------------
    # CoxNet
    # ---------------------------

    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.01,
    },

    {
        "learner": "coxnet",
        "l1_ratio": 0.9,
        "alpha_min_ratio": 0.01,
    },

    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.1,
    },


    # ---------------------------
    # GBSA
    # ---------------------------

    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2,
        "subsample": 0.8,
    },

    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3,
        "subsample": 0.8,
    },
]


# ======================================================================
# 4. PRINTING / CHECKPOINT UTILITIES
# ======================================================================

def section(title):

    print("\n" + "=" * 80)

    print(title)

    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] "
        f"{message}"
    )

    if not condition:

        raise RuntimeError(
            "Checkpoint failed: "
            + message
        )


# ======================================================================
# 5. ID UTILITIES
# ======================================================================

def normalize_patient_id(x):

    x = str(x).strip()

    if x.startswith("TCGA-"):

        parts = x.split("-")

        if len(parts) >= 3:

            return "-".join(
                parts[:3]
            )

    return x


def strip_version(x):

    return (
        str(x)
        .split(".")[0]
        .strip()
    )


# ======================================================================
# 6. SURVIVAL UTILITIES
# ======================================================================

def make_survival_y(df):

    events = pd.to_numeric(
        df["OS_event"],
        errors="raise",
    ).astype(int)


    times = pd.to_numeric(
        df["OS_months"],
        errors="raise",
    ).astype(float)


    if not events.isin(
        [0, 1]
    ).all():

        raise ValueError(
            "OS_event must contain only 0/1."
        )


    if (
        times <= 0
    ).any():

        raise ValueError(
            "OS_months must be > 0."
        )


    return np.array(

        [
            (
                bool(event),
                float(time_value),
            )

            for event, time_value
            in zip(
                events,
                times,
            )
        ],

        dtype=[
            ("event", "?"),
            ("time", "<f8"),
        ],
    )


def cindex(y, risk):

    risk = np.asarray(
        risk,
        dtype=float,
    )


    if not np.isfinite(
        risk
    ).all():

        return np.nan


    return float(

        concordance_index_censored(

            y["event"],

            y["time"],

            risk,

        )[0]

    )


# ======================================================================
# 7. CHECK FILES
# ======================================================================

section(
    "STEP 1 — CHECK REQUIRED FILES"
)


required_files = [

    SPLIT_FILE,

    COHORT_FILE,

    M2_SAVED_FILE,

    M3_SAVED_FILE,

    ENESH_FULL_FILE,

    EXPRESSION_FULL_FILE,

    V4_SPECS_FILE,
]


for file_path in required_files:

    checkpoint(
        file_path.exists(),
        str(file_path),
    )


# ======================================================================
# 8. LOAD SURVIVAL COHORT
# ======================================================================

section(
    "STEP 2 — LOAD SURVIVAL COHORT"
)


cohort = pd.read_csv(
    COHORT_FILE
)


cohort[
    "patient_id"
] = (
    cohort[
        "patient_id"
    ]
    .apply(
        normalize_patient_id
    )
)


checkpoint(
    len(cohort) == 144,
    "survival cohort contains 144 patients",
)


checkpoint(
    cohort[
        "patient_id"
    ].nunique() == 144,
    "144 unique patient IDs",
)


checkpoint(
    cohort[
        "OS_months"
    ].notna().all(),
    "all survival times present",
)


checkpoint(
    cohort[
        "OS_event"
    ].notna().all(),
    "all event indicators present",
)


survival = (
    cohort
    .set_index(
        "patient_id"
    )
)


print(
    "\nDeaths:",
    int(
        survival[
            "OS_event"
        ].sum()
    )
)


print(
    "Censored:",
    int(
        len(survival)
        -
        survival[
            "OS_event"
        ].sum()
    )
)


# ======================================================================
# 9. LOAD ORIGINAL LOCKED SPLIT
# ======================================================================

section(
    "STEP 3 — LOAD ORIGINAL LOCKED SPLIT"
)


split_df = pd.read_csv(
    SPLIT_FILE
)


split_df[
    "patient_id"
] = (
    split_df[
        "patient_id"
    ]
    .apply(
        normalize_patient_id
    )
)


original_dev_ids = (

    split_df.loc[

        split_df[
            "split"
        ]
        == "development",

        "patient_id",

    ]
    .tolist()

)


original_test_ids = (

    split_df.loc[

        split_df[
            "split"
        ]
        == "locked_test",

        "patient_id",

    ]
    .tolist()

)


checkpoint(
    len(
        original_dev_ids
    ) == 115,
    "original development cohort = 115",
)


checkpoint(
    len(
        original_test_ids
    ) == 29,
    "original internal test = 29",
)


# ======================================================================
# 10. LOAD ORIGINAL FULL 3,992 eNESH MATRIX
# ======================================================================

section(
    "STEP 4 — LOAD FULL eNESH MATRIX"
)


enesh_full = pd.read_csv(
    ENESH_FULL_FILE,
    index_col=0,
)


enesh_full.index = [

    normalize_patient_id(x)

    for x in enesh_full.index

]


enesh_full.columns = [

    strip_version(x)

    for x in enesh_full.columns

]


checkpoint(
    enesh_full.shape[0] == 144,
    "eNESH contains 144 patients",
)


checkpoint(
    enesh_full.shape[1] == 3992,
    "eNESH contains original 3,992 anchors",
)


checkpoint(
    enesh_full.index.is_unique,
    "eNESH patient IDs unique",
)


checkpoint(
    enesh_full.columns.is_unique,
    "eNESH gene IDs unique",
)


checkpoint(
    not enesh_full.isna().any().any(),
    "eNESH contains no missing values",
)


checkpoint(
    np.isfinite(
        enesh_full.to_numpy(
            dtype=float
        )
    ).all(),
    "all eNESH values finite",
)


# ======================================================================
# 11. LOAD FULL CALIBRATED EXPRESSION MATRIX
# ======================================================================

section(
    "STEP 5 — LOAD CALIBRATED EXPRESSION"
)


expression_raw = pd.read_csv(
    EXPRESSION_FULL_FILE,
    index_col=0,
)


expression_raw.index = [

    strip_version(x)

    for x in expression_raw.index

]


patient_set = set(
    survival.index
)


column_patient_ids = set(

    normalize_patient_id(x)

    for x in expression_raw.columns

)


index_patient_ids = set(

    normalize_patient_id(x)

    for x in expression_raw.index

)


if len(
    column_patient_ids
    & patient_set
) == 144:

    print(
        "Expression orientation: genes × patients"
    )

    expression_raw.columns = [

        normalize_patient_id(x)

        for x in expression_raw.columns

    ]

    expression_full = (
        expression_raw.T
    )


elif len(
    index_patient_ids
    & patient_set
) == 144:

    print(
        "Expression orientation: patients × genes"
    )

    expression_raw.index = [

        normalize_patient_id(x)

        for x in expression_raw.index

    ]

    expression_full = (
        expression_raw.copy()
    )


else:

    raise RuntimeError(
        "Could not determine expression matrix orientation."
    )


expression_full.columns = [

    strip_version(x)

    for x in expression_full.columns

]


checkpoint(
    set(
        enesh_full.index
    )
    == set(
        expression_full.index
    ),
    "expression and eNESH contain same 144 patients",
)


checkpoint(
    set(
        enesh_full.columns
    ).issubset(
        set(
            expression_full.columns
        )
    ),
    "all 3,992 eNESH anchors available in expression",
)


# Use exact anchor universe/order from eNESH

expression_full = (

    expression_full
    .loc[
        enesh_full.index,
        enesh_full.columns,
    ]
    .copy()

)


checkpoint(
    expression_full.shape
    == (
        144,
        3992,
    ),
    "full expression matrix = 144 × 3,992",
)


checkpoint(
    not expression_full.isna().any().any(),
    "expression contains no missing values",
)


checkpoint(
    np.isfinite(
        expression_full.to_numpy(
            dtype=float
        )
    ).all(),
    "all expression values finite",
)


# ======================================================================
# 12. ESTABLISH ORIGINAL STAGE-C PATIENT ORDER
# ======================================================================

section(
    "STEP 6 — ESTABLISH ORIGINAL PATIENT ORDER"
)


patient_ids = np.array(

    enesh_full.index.tolist(),

    dtype=object,

)


checkpoint(
    set(
        patient_ids
    )
    == set(
        survival.index
    ),
    "Stage-C order contains exact survival cohort",
)


event_vector = (

    survival
    .loc[
        patient_ids,
        "OS_event",
    ]
    .astype(int)
    .to_numpy()

)


# ======================================================================
# 13. SPLIT FUNCTION — CORRECTED ORDER HANDLING
# ======================================================================

def create_ordered_split(
    split_seed,
):

    """
    Reproduce train/test membership using sklearn, then restore
    the original Stage-C patient ordering.

    This is critical because the original Stage-D split file retained
    source-data order rather than train_test_split's shuffled output
    order.
    """


    raw_development_ids, \
    raw_test_ids = (
        train_test_split(

            patient_ids,

            test_size=0.20,

            random_state=
                int(
                    split_seed
                ),

            stratify=
                event_vector,

        )
    )


    development_set = set(
        raw_development_ids
    )


    test_set = set(
        raw_test_ids
    )


    development_ids = [

        patient_id

        for patient_id
        in patient_ids

        if patient_id
        in development_set

    ]


    test_ids = [

        patient_id

        for patient_id
        in patient_ids

        if patient_id
        in test_set

    ]


    return (
        development_ids,
        test_ids,
    )


# ======================================================================
# 14. EXACT SEED-42 SPLIT AUDIT
# ======================================================================

section(
    "STEP 7 — REGENERATE ORIGINAL SEED-42 SPLIT"
)


seed42_dev_ids, \
seed42_test_ids = (
    create_ordered_split(
        42
    )
)


checkpoint(
    seed42_dev_ids
    == original_dev_ids,
    (
        "seed-42 development IDs AND ORDER "
        "match original locked split exactly"
    ),
)


checkpoint(
    seed42_test_ids
    == original_test_ids,
    (
        "seed-42 test IDs AND ORDER "
        "match original locked split exactly"
    ),
)


print(
    "\nSeed-42 membership and ordering successfully reconstructed."
)


# ======================================================================
# 15. DEVELOPMENT-ONLY ZERO-VARIANCE FILTER
# ======================================================================

def build_split_feature_space(
    development_ids,
):

    """
    Reproduce Stage-D molecular feature-space construction.

    1. Start from 3,992 eNESH anchors.
    2. Calculate variance using DEVELOPMENT patients only.
    3. Remove eNESH anchors with variance <= 1e-10.
    4. Restrict expression to the exact same retained genes.
    """


    development_enesh = (

        enesh_full
        .loc[
            development_ids
        ]

    )


    variances = (

        development_enesh
        .var(
            axis=0,
            ddof=1,
        )

    )


    usable_genes = (

        variances[
            variances
            > ZERO_VARIANCE_THRESHOLD
        ]
        .index
        .tolist()

    )


    rewiring = (

        enesh_full
        .loc[
            :,
            usable_genes,
        ]
        .copy()

    )


    expression = (

        expression_full
        .loc[
            :,
            usable_genes,
        ]
        .copy()

    )


    return (
        expression,
        rewiring,
        usable_genes,
    )


# ======================================================================
# 16. RECONSTRUCT ORIGINAL 3,669 FEATURE SPACE
# ======================================================================

section(
    "STEP 8 — RECONSTRUCT ORIGINAL 3,669-FEATURE SPACE"
)


seed42_expression, \
seed42_rewiring, \
seed42_usable_genes = (
    build_split_feature_space(
        seed42_dev_ids
    )
)


print(
    "\nOriginal seed-42 retained features:",
    len(
        seed42_usable_genes
    ),
)


checkpoint(
    len(
        seed42_usable_genes
    ) == 3669,
    (
        "development-only zero-variance filter "
        "reproduces 3,669 features"
    ),
)


# ======================================================================
# 17. LOAD SAVED STAGE-D M2/M3 MATRICES
# ======================================================================

saved_m2 = pd.read_csv(
    M2_SAVED_FILE,
    index_col=0,
)


saved_m3 = pd.read_csv(
    M3_SAVED_FILE,
    index_col=0,
)


saved_m2.index = [

    normalize_patient_id(x)

    for x in saved_m2.index

]


saved_m3.index = [

    normalize_patient_id(x)

    for x in saved_m3.index

]


saved_m2.columns = [

    strip_version(x)

    for x in saved_m2.columns

]


saved_m3.columns = [

    strip_version(x)

    for x in saved_m3.columns

]


checkpoint(
    set(
        seed42_usable_genes
    )
    == set(
        saved_m2.columns
    ),
    "reconstructed genes match saved Stage-D M2",
)


checkpoint(
    set(
        seed42_usable_genes
    )
    == set(
        saved_m3.columns
    ),
    "reconstructed genes match saved Stage-D M3",
)


# ======================================================================
# 18. NUMERICAL MATRIX AUDIT
# ======================================================================

reconstructed_m2 = (

    seed42_expression
    .loc[
        saved_m2.index,
        saved_m2.columns,
    ]

)


reconstructed_m3 = (

    seed42_rewiring
    .loc[
        saved_m3.index,
        saved_m3.columns,
    ]

)


m2_matrix_match = np.allclose(

    reconstructed_m2.to_numpy(
        dtype=float
    ),

    saved_m2.to_numpy(
        dtype=float
    ),

    rtol=1e-10,

    atol=1e-12,

    equal_nan=True,

)


m3_matrix_match = np.allclose(

    reconstructed_m3.to_numpy(
        dtype=float
    ),

    saved_m3.to_numpy(
        dtype=float
    ),

    rtol=1e-10,

    atol=1e-12,

    equal_nan=True,

)


checkpoint(
    m2_matrix_match,
    "reconstructed M2 values match saved Stage-D M2 numerically",
)


checkpoint(
    m3_matrix_match,
    "reconstructed M3 values match saved Stage-D M3 numerically",
)


# ======================================================================
# 19. MOLECULAR FEATURE RANKING
# ======================================================================

def rank_molecular_features(
    X_train,
    y_train,
):

    feature_names = list(
        X_train.columns
    )


    values = X_train.to_numpy(
        dtype=np.float64
    )


    scores = np.full(
        len(
            feature_names
        ),
        0.5,
        dtype=float,
    )


    for j in range(
        values.shape[1]
    ):

        x = values[
            :,
            j
        ]


        if np.nanstd(
            x
        ) <= 1e-12:

            continue


        try:

            raw_c = cindex(
                y_train,
                x,
            )


            if np.isfinite(
                raw_c
            ):

                scores[j] = max(
                    raw_c,
                    1.0 - raw_c,
                )


        except Exception:

            scores[j] = 0.5


    ranking = pd.DataFrame(
        {
            "feature":
                feature_names,

            "score":
                scores,
        }
    )


    ranking = (

        ranking
        .sort_values(

            [
                "score",
                "feature",
            ],

            ascending=[
                False,
                True,
            ],

        )
        .reset_index(
            drop=True
        )

    )


    return ranking


# ======================================================================
# 20. FIT/PREDICT BASE LEARNERS
# ======================================================================

def fit_predict(
    params,
    X_train,
    y_train,
    X_valid,
    seed=RANDOM_SEED,
):

    learner = params[
        "learner"
    ]


    # ------------------------------------------------------------------
    # RSF
    # ------------------------------------------------------------------

    if learner == "rsf":

        model = (
            RandomSurvivalForest(

                n_estimators=
                    params[
                        "n_estimators"
                    ],

                min_samples_leaf=
                    params[
                        "min_samples_leaf"
                    ],

                min_samples_split=10,

                max_features=
                    params[
                        "max_features"
                    ],

                random_state=
                    seed,

                n_jobs=-1,

            )
        )


    # ------------------------------------------------------------------
    # CoxNet
    # ------------------------------------------------------------------

    elif learner == "coxnet":

        model = (
            CoxnetSurvivalAnalysis(

                l1_ratio=
                    params[
                        "l1_ratio"
                    ],

                alpha_min_ratio=
                    params[
                        "alpha_min_ratio"
                    ],

            )
        )


    # ------------------------------------------------------------------
    # GBSA
    # ------------------------------------------------------------------

    elif learner == "gbsa":

        model = (
            GradientBoostingSurvivalAnalysis(

                n_estimators=
                    params[
                        "n_estimators"
                    ],

                learning_rate=
                    params[
                        "learning_rate"
                    ],

                max_depth=
                    params[
                        "max_depth"
                    ],

                subsample=
                    params[
                        "subsample"
                    ],

                random_state=
                    seed,

            )
        )


    else:

        raise ValueError(
            f"Unknown learner: {learner}"
        )


    model.fit(
        X_train,
        y_train,
    )


    risk = model.predict(
        X_valid
    )


    return risk


# ======================================================================
# 21. FINAL SEED-ENSEMBLE FITTING
# ======================================================================

def fit_predict_final(
    params,
    X_train,
    y_train,
    X_test,
):

    """
    V4 final fitting rule.

    RSF:
        average predictions from random states 42,43,44,45,46.

    Non-RSF:
        ordinary final fit.
    """


    if params[
        "learner"
    ] != "rsf":

        return fit_predict(

            params,

            X_train,

            y_train,

            X_test,

            seed=RANDOM_SEED,

        )


    predictions = []


    for seed in range(
        RANDOM_SEED,
        RANDOM_SEED
        + N_FINAL_SEEDS,
    ):

        prediction = (
            fit_predict(

                params,

                X_train,

                y_train,

                X_test,

                seed=seed,

            )
        )


        predictions.append(
            prediction
        )


    return np.mean(
        np.vstack(
            predictions
        ),
        axis=0,
    )


# ======================================================================
# 22. FEATURE SELECTION
# ======================================================================

def select_features(
    molecular,
    training_ids,
    y_training,
    top_k,
):

    ranking = (
        rank_molecular_features(

            molecular.loc[
                training_ids
            ],

            y_training,

        )
    )


    selected = (

        ranking[
            "feature"
        ]
        .head(
            int(
                top_k
            )
        )
        .tolist()

    )


    return (
        selected,
        ranking,
    )


# ======================================================================
# 23. EVALUATE ONE MOLECULAR SPECIFICATION
# ======================================================================

def evaluate_molecular_model(

    molecular,

    training_ids,

    validation_ids,

    top_k,

    params,

    final_seed_ensemble=False,

):

    y_training = (
        make_survival_y(

            survival.loc[
                training_ids
            ]

        )
    )


    selected, ranking = (
        select_features(

            molecular,

            training_ids,

            y_training,

            top_k,

        )
    )


    X_training = (

        molecular
        .loc[
            training_ids,
            selected,
        ]
        .to_numpy(
            dtype=np.float64
        )

    )


    X_validation = (

        molecular
        .loc[
            validation_ids,
            selected,
        ]
        .to_numpy(
            dtype=np.float64
        )

    )


    if final_seed_ensemble:

        risk = (
            fit_predict_final(

                params,

                X_training,

                y_training,

                X_validation,

            )
        )


    else:

        risk = (
            fit_predict(

                params,

                X_training,

                y_training,

                X_validation,

                seed=RANDOM_SEED,

            )
        )


    return (
        risk,
        selected,
        ranking,
    )


# ======================================================================
# 24. V4 FINAL 3-FOLD TUNING
# ======================================================================

def tune_molecular_model(
    molecular,
    development_ids,
):

    """
    Final V4 tuning:

        3-fold event-stratified CV
        random_state = 1042

    Within every training fold:
        molecular ranking is recalculated.

    Search:
        K = 10, 25, 50
        ×
        all learner specifications.
    """


    development_ids = list(
        development_ids
    )


    development_events = (

        survival
        .loc[
            development_ids,
            "OS_event",
        ]
        .astype(int)
        .to_numpy()

    )


    candidate_results = []


    for top_k in TOP_K_VALUES:

        for params in LEARNER_GRID:

            fold_scores = []


            cv = (
                StratifiedKFold(

                    n_splits=
                        INNER_FOLDS,

                    shuffle=True,

                    random_state=
                        RANDOM_SEED
                        + 1000,

                )
            )


            for (
                train_index,
                validation_index,
            ) in cv.split(

                np.arange(
                    len(
                        development_ids
                    )
                ),

                development_events,

            ):


                inner_train_ids = [

                    development_ids[i]

                    for i
                    in train_index

                ]


                inner_validation_ids = [

                    development_ids[i]

                    for i
                    in validation_index

                ]


                y_validation = (
                    make_survival_y(

                        survival.loc[
                            inner_validation_ids
                        ]

                    )
                )


                try:

                    (
                        risk,
                        _,
                        _,
                    ) = evaluate_molecular_model(

                        molecular,

                        inner_train_ids,

                        inner_validation_ids,

                        top_k,

                        params,

                        final_seed_ensemble=False,

                    )


                    score = cindex(

                        y_validation,

                        risk,

                    )


                except Exception:

                    score = np.nan


                fold_scores.append(
                    score
                )


            valid_scores = [

                value

                for value
                in fold_scores

                if np.isfinite(
                    value
                )

            ]


            if len(
                valid_scores
            ) > 0:

                mean_cindex = float(
                    np.mean(
                        valid_scores
                    )
                )


            else:

                mean_cindex = np.nan


            candidate_results.append(
                {
                    "top_k":
                        int(
                            top_k
                        ),

                    "params":
                        params.copy(),

                    "mean_cindex":
                        mean_cindex,

                    "fold_scores":
                        fold_scores,
                }
            )


    valid_candidates = [

        result

        for result
        in candidate_results

        if np.isfinite(
            result[
                "mean_cindex"
            ]
        )

    ]


    if len(
        valid_candidates
    ) == 0:

        raise RuntimeError(
            "No valid model candidates."
        )


    best = max(

        valid_candidates,

        key=lambda result:
            result[
                "mean_cindex"
            ],

    )


    return (
        best,
        candidate_results,
    )


# ======================================================================
# 25. COMPLETE ONE-PARTITION ANALYSIS
# ======================================================================

def run_one_partition(
    split_seed,
    verbose=False,
):

    # ------------------------------------------------------------------
    # Correctly ordered 115/29 split
    # ------------------------------------------------------------------

    development_ids, \
    test_ids = (
        create_ordered_split(
            split_seed
        )
    )


    assert (
        len(
            development_ids
        ) == 115
    )


    assert (
        len(
            test_ids
        ) == 29
    )


    # ------------------------------------------------------------------
    # Rebuild development-dependent molecular universe
    # ------------------------------------------------------------------

    expression, \
    rewiring, \
    usable_genes = (
        build_split_feature_space(
            development_ids
        )
    )


    y_test = (
        make_survival_y(

            survival.loc[
                test_ids
            ]

        )
    )


    # ==================================================================
    # M2 TUNING
    # ==================================================================

    m2_best, \
    m2_candidate_results = (
        tune_molecular_model(

            expression,

            development_ids,

        )
    )


    (
        m2_risk,
        m2_selected,
        m2_final_ranking,
    ) = evaluate_molecular_model(

        expression,

        development_ids,

        test_ids,

        m2_best[
            "top_k"
        ],

        m2_best[
            "params"
        ],

        final_seed_ensemble=True,

    )


    m2_test_c = cindex(
        y_test,
        m2_risk,
    )


    # ==================================================================
    # M3 TUNING
    # ==================================================================

    m3_best, \
    m3_candidate_results = (
        tune_molecular_model(

            rewiring,

            development_ids,

        )
    )


    (
        m3_risk,
        m3_selected,
        m3_final_ranking,
    ) = evaluate_molecular_model(

        rewiring,

        development_ids,

        test_ids,

        m3_best[
            "top_k"
        ],

        m3_best[
            "params"
        ],

        final_seed_ensemble=True,

    )


    m3_test_c = cindex(
        y_test,
        m3_risk,
    )


    if verbose:

        print(
            "\nM2 best specification:"
        )

        print(
            m2_best
        )


        print(
            "\nM3 best specification:"
        )

        print(
            m3_best
        )


        print(
            "\nM2 test C-index:",
            m2_test_c
        )


        print(
            "M3 test C-index:",
            m3_test_c
        )


    result = {

        "split_seed":
            int(
                split_seed
            ),

        "n_development":
            len(
                development_ids
            ),

        "n_test":
            len(
                test_ids
            ),

        "development_events":
            int(
                survival
                .loc[
                    development_ids,
                    "OS_event",
                ]
                .sum()
            ),

        "test_events":
            int(
                survival
                .loc[
                    test_ids,
                    "OS_event",
                ]
                .sum()
            ),

        "n_usable_genes":
            len(
                usable_genes
            ),


        # --------------------------------------------------------------
        # M2
        # --------------------------------------------------------------

        "M2_inner_C":
            float(
                m2_best[
                    "mean_cindex"
                ]
            ),

        "M2_test_C":
            float(
                m2_test_c
            ),

        "M2_top_k":
            int(
                m2_best[
                    "top_k"
                ]
            ),

        "M2_learner":
            m2_best[
                "params"
            ][
                "learner"
            ],

        "M2_params":
            json.dumps(
                m2_best[
                    "params"
                ],
                sort_keys=True,
            ),


        # --------------------------------------------------------------
        # M3
        # --------------------------------------------------------------

        "M3_inner_C":
            float(
                m3_best[
                    "mean_cindex"
                ]
            ),

        "M3_test_C":
            float(
                m3_test_c
            ),

        "M3_top_k":
            int(
                m3_best[
                    "top_k"
                ]
            ),

        "M3_learner":
            m3_best[
                "params"
            ][
                "learner"
            ],

        "M3_params":
            json.dumps(
                m3_best[
                    "params"
                ],
                sort_keys=True,
            ),

    }


    return {

        "result":
            result,

        "m2_selected":
            m2_selected,

        "m3_selected":
            m3_selected,

        "usable_genes":
            usable_genes,

        "m2_candidates":
            m2_candidate_results,

        "m3_candidates":
            m3_candidate_results,

    }


# ======================================================================
# 26. LOAD SAVED V4 SPECIFICATIONS
# ======================================================================

with open(
    V4_SPECS_FILE,
    "r",
) as file:

    saved_specs = json.load(
        file
    )


section(
    "STEP 9 — SAVED ORIGINAL V4 SPECIFICATIONS"
)


print(
    "M2:"
)

print(
    saved_specs[
        "M2"
    ]
)


print(
    "\nM3:"
)

print(
    saved_specs[
        "M3"
    ]
)


# ======================================================================
# 27. CRITICAL SEED-42 RECONSTRUCTION
# ======================================================================

section(
    "STEP 10 — EXACT SEED-42 V4 RECONSTRUCTION"
)


print(
    "Running complete V4 final tuning for seed 42..."
)


audit_start = time.time()


seed42_output = (
    run_one_partition(

        42,

        verbose=True,

    )
)


seed42_result = (
    seed42_output[
        "result"
    ]
)


audit_minutes = (

    time.time()
    - audit_start

) / 60


print(
    "\nSeed-42 audit runtime:",
    round(
        audit_minutes,
        2,
    ),
    "minutes",
)


# ======================================================================
# 28. PRINT RECONSTRUCTION
# ======================================================================

section(
    "SEED-42 RECONSTRUCTION SUMMARY"
)


print(
    "Usable genes:"
)


print(
    "  Reconstructed:",
    seed42_result[
        "n_usable_genes"
    ]
)


print(
    "  Expected:     3669"
)


print(
    "\nM2 reconstructed:"
)


print(
    "  learner:",
    seed42_result[
        "M2_learner"
    ]
)


print(
    "  K:",
    seed42_result[
        "M2_top_k"
    ]
)


print(
    "  inner C:",
    seed42_result[
        "M2_inner_C"
    ]
)


print(
    "  test C:",
    seed42_result[
        "M2_test_C"
    ]
)


print(
    "\nM2 saved:"
)


print(
    saved_specs[
        "M2"
    ]
)


print(
    "  manuscript test C:",
    ORIGINAL_M2_C
)


print(
    "\nM3 reconstructed:"
)


print(
    "  learner:",
    seed42_result[
        "M3_learner"
    ]
)


print(
    "  K:",
    seed42_result[
        "M3_top_k"
    ]
)


print(
    "  inner C:",
    seed42_result[
        "M3_inner_C"
    ]
)


print(
    "  test C:",
    seed42_result[
        "M3_test_C"
    ]
)


print(
    "\nM3 saved:"
)


print(
    saved_specs[
        "M3"
    ]
)


print(
    "  exact original test C:",
    ORIGINAL_M3_C_EXACT
)


# ======================================================================
# 29. SPECIFICATION COMPARISON UTILITIES
# ======================================================================

def normalize_value(value):

    if isinstance(
        value,
        np.generic
    ):

        return value.item()

    return value


def params_equal(
    reconstructed,
    saved,
):

    if set(
        reconstructed.keys()
    ) != set(
        saved.keys()
    ):

        return False


    for key in reconstructed:

        value_a = normalize_value(
            reconstructed[
                key
            ]
        )


        value_b = normalize_value(
            saved[
                key
            ]
        )


        if isinstance(
            value_a,
            float
        ) or isinstance(
            value_b,
            float
        ):

            try:

                if not np.isclose(
                    float(
                        value_a
                    ),
                    float(
                        value_b
                    ),
                    rtol=1e-12,
                    atol=1e-12,
                ):

                    return False


            except Exception:

                if value_a != value_b:

                    return False


        else:

            if value_a != value_b:

                return False


    return True


# ======================================================================
# 30. HARD SEED-42 AUDIT
# ======================================================================

section(
    "STEP 11 — HARD REPRODUCIBILITY AUDIT"
)


# ----------------------------------------------------------------------
# M2 Top-K
# ----------------------------------------------------------------------

checkpoint(

    int(
        seed42_result[
            "M2_top_k"
        ]
    )
    ==
    int(
        saved_specs[
            "M2"
        ][
            "top_k"
        ]
    ),

    "M2 Top-K matches saved V4 specification",

)


# ----------------------------------------------------------------------
# M2 parameters
# ----------------------------------------------------------------------

reconstructed_m2_params = json.loads(

    seed42_result[
        "M2_params"
    ]

)


checkpoint(

    params_equal(

        reconstructed_m2_params,

        saved_specs[
            "M2"
        ][
            "params"
        ],

    ),

    "M2 learner/hyperparameters match saved V4 specification",

)


# ----------------------------------------------------------------------
# M3 Top-K
# ----------------------------------------------------------------------

checkpoint(

    int(
        seed42_result[
            "M3_top_k"
        ]
    )
    ==
    int(
        saved_specs[
            "M3"
        ][
            "top_k"
        ]
    ),

    "M3 Top-K matches saved V4 specification",

)


# ----------------------------------------------------------------------
# M3 parameters
# ----------------------------------------------------------------------

reconstructed_m3_params = json.loads(

    seed42_result[
        "M3_params"
    ]

)


checkpoint(

    params_equal(

        reconstructed_m3_params,

        saved_specs[
            "M3"
        ][
            "params"
        ],

    ),

    "M3 learner/hyperparameters match saved V4 specification",

)


# ----------------------------------------------------------------------
# Inner-C reproduction
# ----------------------------------------------------------------------

m2_inner_difference = abs(

    seed42_result[
        "M2_inner_C"
    ]

    -

    float(
        saved_specs[
            "M2"
        ][
            "mean_cindex"
        ]
    )

)


m3_inner_difference = abs(

    seed42_result[
        "M3_inner_C"
    ]

    -

    float(
        saved_specs[
            "M3"
        ][
            "mean_cindex"
        ]
    )

)


print(
    "\nM2 inner-C difference:",
    m2_inner_difference
)


print(
    "M3 inner-C difference:",
    m3_inner_difference
)


# These should ideally be essentially zero.

checkpoint(

    m2_inner_difference
    < 1e-8,

    "M2 tuning C-index reproduces saved V4 value",

)


checkpoint(

    m3_inner_difference
    < 1e-8,

    "M3 tuning C-index reproduces saved V4 value",

)


# ----------------------------------------------------------------------
# Test-C reproduction
# ----------------------------------------------------------------------

m2_test_difference = abs(

    seed42_result[
        "M2_test_C"
    ]

    -

    ORIGINAL_M2_C

)


m3_test_difference = abs(

    seed42_result[
        "M3_test_C"
    ]

    -

    ORIGINAL_M3_C_EXACT

)


print(
    "\nM2 test-C difference:",
    m2_test_difference
)


print(
    "M3 test-C difference:",
    m3_test_difference
)


# M2 manuscript number is rounded, so use a small tolerance.

checkpoint(

    m2_test_difference
    < 0.005,

    "M2 internal-test C-index reproduces manuscript value",

)


checkpoint(

    m3_test_difference
    < 1e-8,

    "M3 internal-test C-index reproduces exact original value",

)


# ======================================================================
# 31. AUDIT SUCCESS
# ======================================================================

section(
    "ALL SEED-42 AUDITS PASSED"
)


print(
    "The reconstructed pipeline matches the original V4 analysis."
)


print(
    "\nProceeding automatically to the "
    f"{N_SPLITS}-partition sensitivity analysis."
)


# ======================================================================
# 32. RUN COMPLETE 100-PARTITION ANALYSIS
# ======================================================================

section(
    f"STEP 12 — RUN {N_SPLITS} COMPLETE ALTERNATIVE PARTITIONS"
)


all_results = []

m2_feature_records = []

m3_feature_records = []

usable_gene_records = []


analysis_start = time.time()


for split_seed in range(
    N_SPLITS
):

    split_start = time.time()


    output = (
        run_one_partition(
            split_seed,
            verbose=False,
        )
    )


    result = output[
        "result"
    ]


    all_results.append(
        result
    )


    # ------------------------------------------------------------------
    # M2 final selected features
    # ------------------------------------------------------------------

    for rank, feature in enumerate(

        output[
            "m2_selected"
        ],

        start=1,

    ):

        m2_feature_records.append(
            {
                "split_seed":
                    split_seed,

                "rank":
                    rank,

                "feature":
                    feature,
            }
        )


    # ------------------------------------------------------------------
    # M3 final selected features
    # ------------------------------------------------------------------

    for rank, feature in enumerate(

        output[
            "m3_selected"
        ],

        start=1,

    ):

        m3_feature_records.append(
            {
                "split_seed":
                    split_seed,

                "rank":
                    rank,

                "feature":
                    feature,
            }
        )


    # ------------------------------------------------------------------
    # Development-retained feature universe
    # ------------------------------------------------------------------

    for feature in output[
        "usable_genes"
    ]:

        usable_gene_records.append(
            {
                "split_seed":
                    split_seed,

                "feature":
                    feature,
            }
        )


    split_minutes = (

        time.time()
        - split_start

    ) / 60


    elapsed_minutes = (

        time.time()
        - analysis_start

    ) / 60


    completed = (
        split_seed
        + 1
    )


    average_minutes = (
        elapsed_minutes
        / completed
    )


    remaining_minutes = (

        average_minutes

        * (
            N_SPLITS
            - completed
        )

    )


    print(

        f"Split {split_seed:03d} | "

        f"genes={result['n_usable_genes']} | "

        f"M2={result['M2_test_C']:.4f} "
        f"[{result['M2_learner']}, "
        f"K={result['M2_top_k']}] | "

        f"M3={result['M3_test_C']:.4f} "
        f"[{result['M3_learner']}, "
        f"K={result['M3_top_k']}] | "

        f"split={split_minutes:.1f} min | "

        f"remaining≈{remaining_minutes/60:.1f} h"

    )


    # ------------------------------------------------------------------
    # SAVE CHECKPOINT AFTER EVERY COMPLETED PARTITION
    # ------------------------------------------------------------------

    pd.DataFrame(
        all_results
    ).to_csv(

        OUTPUT_DIR
        / "EXACT_V4_SPLIT_SENSITIVITY_CHECKPOINT.csv",

        index=False,

    )


# ======================================================================
# 33. RESULT DATAFRAMES
# ======================================================================

results = pd.DataFrame(
    all_results
)


m2_feature_df = pd.DataFrame(
    m2_feature_records
)


m3_feature_df = pd.DataFrame(
    m3_feature_records
)


usable_gene_df = pd.DataFrame(
    usable_gene_records
)


# ======================================================================
# 34. DISTRIBUTION SUMMARY
# ======================================================================

def summarize_distribution(
    values,
    original_value,
    model_name,
):

    values = pd.Series(
        values,
        dtype=float,
    )


    return {

        "Model":
            model_name,

        "N_splits":
            len(
                values
            ),

        "Mean":
            values.mean(),

        "SD":
            values.std(
                ddof=1
            ),

        "Median":
            values.median(),

        "Q1":
            values.quantile(
                0.25
            ),

        "Q3":
            values.quantile(
                0.75
            ),

        "Min":
            values.min(),

        "Max":
            values.max(),

        "P2.5":
            values.quantile(
                0.025
            ),

        "P97.5":
            values.quantile(
                0.975
            ),

        "Original_C":
            original_value,

        "Original_percentile":
            100
            * np.mean(
                values
                <= original_value
            ),

        "Proportion_ge_original":
            np.mean(
                values
                >= original_value
            ),

        "Proportion_gt_0.5":
            np.mean(
                values
                > 0.5
            ),

    }


summary = pd.DataFrame(

    [

        summarize_distribution(

            results[
                "M2_test_C"
            ],

            ORIGINAL_M2_C,

            "M2 expression",

        ),

        summarize_distribution(

            results[
                "M3_test_C"
            ],

            ORIGINAL_M3_C_EXACT,

            "M3 eNESH",

        ),

    ]

)


# ======================================================================
# 35. MODEL-SELECTION STABILITY
# ======================================================================

m2_model_frequency = (

    results
    .groupby(
        [
            "M2_learner",
            "M2_top_k",
            "M2_params",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )

)


m2_model_frequency[
    "frequency"
] = (

    m2_model_frequency[
        "count"
    ]
    / N_SPLITS

)


m3_model_frequency = (

    results
    .groupby(
        [
            "M3_learner",
            "M3_top_k",
            "M3_params",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )

)


m3_model_frequency[
    "frequency"
] = (

    m3_model_frequency[
        "count"
    ]
    / N_SPLITS

)


# ======================================================================
# 36. FEATURE-SELECTION STABILITY
# ======================================================================

m2_feature_frequency = (

    m2_feature_df[
        "feature"
    ]
    .value_counts()
    .rename_axis(
        "feature"
    )
    .reset_index(
        name="selection_count"
    )

)


m2_feature_frequency[
    "selection_frequency"
] = (

    m2_feature_frequency[
        "selection_count"
    ]
    / N_SPLITS

)


m3_feature_frequency = (

    m3_feature_df[
        "feature"
    ]
    .value_counts()
    .rename_axis(
        "feature"
    )
    .reset_index(
        name="selection_count"
    )

)


m3_feature_frequency[
    "selection_frequency"
] = (

    m3_feature_frequency[
        "selection_count"
    ]
    / N_SPLITS

)


# ======================================================================
# 37. FEATURE-UNIVERSE STABILITY
# ======================================================================

usable_gene_frequency = (

    usable_gene_df[
        "feature"
    ]
    .value_counts()
    .rename_axis(
        "feature"
    )
    .reset_index(
        name="retained_count"
    )

)


usable_gene_frequency[
    "retention_frequency"
] = (

    usable_gene_frequency[
        "retained_count"
    ]
    / N_SPLITS

)


# ======================================================================
# 38. PRINT FINAL RESULTS
# ======================================================================

section(
    "FINAL EXACT V4 PARTITION-SENSITIVITY RESULTS"
)


print(
    summary.to_string(
        index=False
    )
)


section(
    "M2 MODEL-SELECTION FREQUENCY"
)


print(
    m2_model_frequency.to_string(
        index=False
    )
)


section(
    "M3 MODEL-SELECTION FREQUENCY"
)


print(
    m3_model_frequency.to_string(
        index=False
    )
)


section(
    "TOP 20 M2 FINAL FEATURES"
)


print(
    m2_feature_frequency
    .head(20)
    .to_string(
        index=False
    )
)


section(
    "TOP 20 M3 FINAL FEATURES"
)


print(
    m3_feature_frequency
    .head(20)
    .to_string(
        index=False
    )
)


# ======================================================================
# 39. SAVE CSV FILES
# ======================================================================

results.to_csv(

    OUTPUT_DIR
    / "EXACT_V4_M2_M3_100_SPLIT_RESULTS.csv",

    index=False,

)


summary.to_csv(

    OUTPUT_DIR
    / "EXACT_V4_M2_M3_100_SPLIT_SUMMARY.csv",

    index=False,

)


m2_model_frequency.to_csv(

    OUTPUT_DIR
    / "M2_MODEL_SELECTION_FREQUENCY.csv",

    index=False,

)


m3_model_frequency.to_csv(

    OUTPUT_DIR
    / "M3_MODEL_SELECTION_FREQUENCY.csv",

    index=False,

)


m2_feature_frequency.to_csv(

    OUTPUT_DIR
    / "M2_FINAL_FEATURE_SELECTION_FREQUENCY.csv",

    index=False,

)


m3_feature_frequency.to_csv(

    OUTPUT_DIR
    / "M3_FINAL_FEATURE_SELECTION_FREQUENCY.csv",

    index=False,

)


usable_gene_frequency.to_csv(

    OUTPUT_DIR
    / "FEATURE_UNIVERSE_RETENTION_FREQUENCY.csv",

    index=False,

)


# ======================================================================
# 40. M2 HISTOGRAM
# ======================================================================

plt.figure(
    figsize=(8, 5)
)


plt.hist(
    results[
        "M2_test_C"
    ],
    bins=15,
)


plt.axvline(
    ORIGINAL_M2_C,
    linestyle="--",
    linewidth=2,
    label=(
        f"Original C = "
        f"{ORIGINAL_M2_C:.3f}"
    ),
)


plt.axvline(
    0.5,
    linestyle=":",
    linewidth=2,
    label="C = 0.5",
)


plt.xlabel(
    "Internal-test C-index"
)


plt.ylabel(
    "Number of alternative partitions"
)


plt.title(
    "M2 expression: sensitivity to TCGA 80:20 partition"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR
    / "EXACT_M2_SPLIT_SENSITIVITY.png",

    dpi=300,

    bbox_inches="tight",

)


plt.show()


# ======================================================================
# 41. M3 HISTOGRAM
# ======================================================================

plt.figure(
    figsize=(8, 5)
)


plt.hist(
    results[
        "M3_test_C"
    ],
    bins=15,
)


plt.axvline(
    ORIGINAL_M3_C_EXACT,
    linestyle="--",
    linewidth=2,
    label=(
        f"Original C = "
        f"{ORIGINAL_M3_C_EXACT:.3f}"
    ),
)


plt.axvline(
    0.5,
    linestyle=":",
    linewidth=2,
    label="C = 0.5",
)


plt.xlabel(
    "Internal-test C-index"
)


plt.ylabel(
    "Number of alternative partitions"
)


plt.title(
    "M3 eNESH: sensitivity to TCGA 80:20 partition"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR
    / "EXACT_M3_SPLIT_SENSITIVITY.png",

    dpi=300,

    bbox_inches="tight",

)


plt.show()


# ======================================================================
# 42. M2 VS M3 SCATTER
# ======================================================================

plt.figure(
    figsize=(6, 6)
)


plt.scatter(

    results[
        "M2_test_C"
    ],

    results[
        "M3_test_C"
    ],

    alpha=0.7,

)


minimum_value = min(

    results[
        "M2_test_C"
    ].min(),

    results[
        "M3_test_C"
    ].min(),

)


maximum_value = max(

    results[
        "M2_test_C"
    ].max(),

    results[
        "M3_test_C"
    ].max(),

)


plt.plot(

    [
        minimum_value,
        maximum_value,
    ],

    [
        minimum_value,
        maximum_value,
    ],

    linestyle="--",

)


plt.xlabel(
    "M2 expression C-index"
)


plt.ylabel(
    "M3 eNESH C-index"
)


plt.title(
    "M2 versus M3 across alternative TCGA partitions"
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR
    / "EXACT_M2_VS_M3_SPLIT_SENSITIVITY.png",

    dpi=300,

    bbox_inches="tight",

)


plt.show()


# ======================================================================
# 43. FINAL RUNTIME / OUTPUT
# ======================================================================

section(
    "ANALYSIS COMPLETE"
)


total_minutes = (

    time.time()
    - analysis_start

) / 60


print(
    "100-partition runtime:",
    round(
        total_minutes,
        2,
    ),
    "minutes"
)


print(
    "\nOutput directory:"
)


print(
    OUTPUT_DIR
)


print(
    "\nMain summary:"
)


print(
    OUTPUT_DIR
    / "EXACT_V4_M2_M3_100_SPLIT_SUMMARY.csv"
)


print(
    "\nMain individual-split results:"
)


print(
    OUTPUT_DIR
    / "EXACT_V4_M2_M3_100_SPLIT_RESULTS.csv"
)


print(
    "\nCHECK THESE OUTPUT SECTIONS:"
)


print(
    "1. SEED-42 RECONSTRUCTION SUMMARY"
)


print(
    "2. STEP 11 — HARD REPRODUCIBILITY AUDIT"
)


print(
    "3. FINAL EXACT V4 PARTITION-SENSITIVITY RESULTS"
)


print(
    "4. M2 MODEL-SELECTION FREQUENCY"
)


print(
    "5. M3 MODEL-SELECTION FREQUENCY"
)


print(
    "6. TOP 20 M2 FINAL FEATURES"
)


print(
    "7. TOP 20 M3 FINAL FEATURES"
)


print(
    "\nAlso keep the three PNG figures."
)

### FAST FIXED-SPECIFICATION REPEATED 80:20 PARTITION-SENSITIVITY ANALYSIS

In [ ]:
# ======================================================================
# FAST FIXED-SPECIFICATION REPEATED 80:20 PARTITION-SENSITIVITY ANALYSIS
# ======================================================================
#
# Design (matches the revised Methods):
#   For each of 100 event-stratified 80:20 partitions of the 144-patient
#   TCGA cohort:
#     1. zero-variance filtering on the 115 development patients only
#     2. molecular feature ranking on the 115 development patients only
#     3. the FINAL PRESPECIFIED V4 M2/M3 specifications (learner, K,
#        hyperparameters) are fitted -- no retuning
#     4. evaluation on the 29-patient holdout
#
# Why it is fast:
#   - no 81-fit tuning loop per model per split (the 20-min step)
#   - univariate C-index ranking is vectorized (all features at once)
#   - partitions run in parallel
#
# Robustness:
#   - seed-42 split, feature space and saved matrices are audited
#   - vectorized ranking is checked against the original sksurv loop
#   - the saved V4 specs must reproduce the original internal-test
#     C-indices (M2 0.6433, M3 0.640127) before the 100 splits run
# ======================================================================

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
RANDOM_SEED = 42
N_FINAL_SEEDS = 5            # RSF final fit: average of seeds 42..46 (V4 rule)
ZERO_VAR = 1e-10
N_SPLITS = 100
N_JOBS = -1                  # set to 1 if parallel causes problems
TIED_TOL = 1e-8              # sksurv default

ORIGINAL_M2_C = 0.6433               # manuscript value (rounded)
ORIGINAL_M3_C_EXACT = 0.640127388535

# ----------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------
DL = BASE
STAGE_D = DL / "TCGA_GBM_STAGE_D_SURVIVAL"
SPLIT_FILE = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
COHORT_FILE = STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
M2_SAVED_FILE = STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv"
M3_SAVED_FILE = STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv"
ENESH_FILE = (DL / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
              / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv")
EXPR_FILE = (DL / "TCGA_GBM_STAGE_C_144_CALIBRATED"
             / "STAGE_C_144_CALIBRATED_DEG_ANCHORS.csv")
SPECS_FILE = STAGE_D / "STAGE_D_PART3_V4_OOF_CORRECTED" / "STAGE_D_V4_OOF_FINAL_SPECS.json"
OUT = STAGE_D / "REVISION_FIXED_SPEC_SPLIT_SENSITIVITY"
OUT.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Utilities
# ----------------------------------------------------------------------
def section(title):
    print("\n" + "=" * 78 + "\n" + title + "\n" + "=" * 78)


def check(cond, msg):
    print(f"[{'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise RuntimeError("Checkpoint failed: " + msg)


def norm_pid(x):
    x = str(x).strip()
    if x.startswith("TCGA-"):
        p = x.split("-")
        if len(p) >= 3:
            return "-".join(p[:3])
    return x


def strip_version(x):
    return str(x).split(".")[0].strip()


def make_y(df):
    e = pd.to_numeric(df["OS_event"], errors="raise").astype(int).to_numpy()
    t = pd.to_numeric(df["OS_months"], errors="raise").astype(float).to_numpy()
    if not np.isin(e, [0, 1]).all() or (t <= 0).any():
        raise ValueError("Invalid survival data.")
    return np.array(list(zip(e.astype(bool), t)), dtype=[("event", "?"), ("time", "<f8")])


def cindex(y, risk):
    risk = np.asarray(risk, dtype=float)
    if not np.isfinite(risk).all():
        return np.nan
    return float(concordance_index_censored(y["event"], y["time"], risk)[0])


# ----------------------------------------------------------------------
# Vectorized univariate C-index ranking (same pair rules as sksurv)
# ----------------------------------------------------------------------
def comparable_pairs(t, e):
    """Pair (i, j) is comparable if i had an event and j survived longer,
    or j was censored at the same time (sksurv convention)."""
    ti, tj = t[:, None], t[None, :]
    ei, ej = e[:, None], e[None, :]
    mask = ei & ((tj > ti) | ((tj == ti) & ~ej))
    return np.nonzero(mask)


def rank_features(X, names, y, block=512):
    I, J = comparable_pairs(y["time"], y["event"])
    n_pairs = len(I)
    scores = np.full(X.shape[1], 0.5)
    for s in range(0, X.shape[1], block):
        B = X[:, s:s + block]
        d = B[I] - B[J]                       # risk_i - risk_j
        ties = np.abs(d) <= TIED_TOL
        conc = (d > 0) & ~ties
        c = (conc.sum(0) + 0.5 * ties.sum(0)) / n_pairs
        scores[s:s + block] = np.maximum(c, 1.0 - c)
    scores[np.nanstd(X, axis=0) <= 1e-12] = 0.5
    rk = pd.DataFrame({"feature": names, "score": scores})
    return rk.sort_values(["score", "feature"], ascending=[False, True]).reset_index(drop=True)


def rank_features_loop(X, names, y):
    """Original implementation, used only to audit the fast version."""
    scores = np.full(X.shape[1], 0.5)
    for j in range(X.shape[1]):
        x = X[:, j]
        if np.nanstd(x) <= 1e-12:
            continue
        c = cindex(y, x)
        if np.isfinite(c):
            scores[j] = max(c, 1 - c)
    rk = pd.DataFrame({"feature": names, "score": scores})
    return rk.sort_values(["score", "feature"], ascending=[False, True]).reset_index(drop=True)


# ----------------------------------------------------------------------
# Learners (identical settings to V4; RSF n_jobs=1 because splits run
# in parallel -- results are unaffected since random_state is fixed)
# ----------------------------------------------------------------------
def fit_predict(params, Xtr, ytr, Xte, seed):
    L = params["learner"]
    if L == "rsf":
        m = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            min_samples_split=10,
            max_features=params["max_features"],
            random_state=seed, n_jobs=1)
    elif L == "coxnet":
        m = CoxnetSurvivalAnalysis(l1_ratio=params["l1_ratio"],
                                   alpha_min_ratio=params["alpha_min_ratio"])
    elif L == "gbsa":
        m = GradientBoostingSurvivalAnalysis(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            random_state=seed)
    else:
        raise ValueError(L)
    m.fit(Xtr, ytr)
    return m.predict(Xte)


def fit_predict_final(params, Xtr, ytr, Xte):
    if params["learner"] != "rsf":
        return fit_predict(params, Xtr, ytr, Xte, RANDOM_SEED)
    preds = [fit_predict(params, Xtr, ytr, Xte, s)
             for s in range(RANDOM_SEED, RANDOM_SEED + N_FINAL_SEEDS)]
    return np.mean(np.vstack(preds), axis=0)


# ======================================================================
# LOAD DATA
# ======================================================================
section("LOAD DATA")
for f in [SPLIT_FILE, COHORT_FILE, M2_SAVED_FILE, M3_SAVED_FILE, ENESH_FILE, EXPR_FILE, SPECS_FILE]:
    check(f.exists(), f.name)

cohort = pd.read_csv(COHORT_FILE)
cohort["patient_id"] = cohort["patient_id"].apply(norm_pid)
check(len(cohort) == 144 and cohort["patient_id"].nunique() == 144, "144 unique patients")
check(cohort[["OS_months", "OS_event"]].notna().all().all(), "no missing survival data")
survival = cohort.set_index("patient_id")

split_df = pd.read_csv(SPLIT_FILE)
split_df["patient_id"] = split_df["patient_id"].apply(norm_pid)
orig_dev = split_df.loc[split_df["split"] == "development", "patient_id"].tolist()
orig_test = split_df.loc[split_df["split"] == "locked_test", "patient_id"].tolist()
check(len(orig_dev) == 115 and len(orig_test) == 29, "original split 115/29")

enesh = pd.read_csv(ENESH_FILE, index_col=0)
enesh.index = [norm_pid(x) for x in enesh.index]
enesh.columns = [strip_version(x) for x in enesh.columns]
check(enesh.shape == (144, 3992), "eNESH 144 x 3,992")
check(enesh.index.is_unique and enesh.columns.is_unique, "eNESH IDs unique")
check(np.isfinite(enesh.to_numpy(float)).all(), "eNESH finite, no missing")

er = pd.read_csv(EXPR_FILE, index_col=0)
pset = set(survival.index)
if len({norm_pid(c) for c in er.columns} & pset) == 144:
    er.columns = [norm_pid(c) for c in er.columns]
    er.index = [strip_version(x) for x in er.index]
    expr = er.T
elif len({norm_pid(i) for i in er.index} & pset) == 144:
    er.index = [norm_pid(i) for i in er.index]
    expr = er.copy()
else:
    raise RuntimeError("Cannot determine expression orientation.")
expr.columns = [strip_version(x) for x in expr.columns]
check(set(enesh.columns).issubset(expr.columns), "all eNESH anchors in expression")
expr = expr.loc[enesh.index, enesh.columns].astype(float)
check(np.isfinite(expr.to_numpy()).all(), "expression finite, no missing")

patient_ids = np.array(enesh.index.tolist(), dtype=object)
check(set(patient_ids) == pset, "patient sets agree")
events = survival.loc[patient_ids, "OS_event"].astype(int).to_numpy()

with open(SPECS_FILE) as fh:
    specs = json.load(fh)
SPEC_M2, SPEC_M3 = specs["M2"], specs["M3"]
print("\nSaved V4 M2:", SPEC_M2)
print("Saved V4 M3:", SPEC_M3)

# numpy copies for speed (row order = patient_ids)
ENESH_NP = enesh.to_numpy(float)
EXPR_NP = expr.to_numpy(float)
GENES = np.array(enesh.columns)
ROW = {p: i for i, p in enumerate(patient_ids)}


# ======================================================================
# CORE PER-SPLIT FUNCTIONS
# ======================================================================
def ordered_split(seed):
    dev, te = train_test_split(patient_ids, test_size=0.20,
                               random_state=int(seed), stratify=events)
    ds, ts = set(dev), set(te)
    return [p for p in patient_ids if p in ds], [p for p in patient_ids if p in ts]


def usable_mask(dev_ids):
    idx = [ROW[p] for p in dev_ids]
    return ENESH_NP[idx].var(axis=0, ddof=1) > ZERO_VAR


def run_split(seed):
    dev, te = ordered_split(seed)
    di, ti = [ROW[p] for p in dev], [ROW[p] for p in te]
    keep = usable_mask(dev)
    genes = GENES[keep]
    y_dev, y_te = make_y(survival.loc[dev]), make_y(survival.loc[te])

    out = {"split_seed": int(seed),
           "test_events": int(y_te["event"].sum()),
           "n_usable_genes": int(keep.sum()),
           "n_overlap_original_test": len(set(te) & set(orig_test))}
    selected = {}
    for name, M, spec in [("M2", EXPR_NP, SPEC_M2), ("M3", ENESH_NP, SPEC_M3)]:
        Mk = M[:, keep]
        rk = rank_features(Mk[di], genes, y_dev)
        top = rk["feature"].head(int(spec["top_k"])).tolist()
        cols = [int(np.where(genes == g)[0][0]) for g in top]
        risk = fit_predict_final(spec["params"], Mk[di][:, cols], y_dev, Mk[ti][:, cols])
        out[f"{name}_test_C"] = cindex(y_te, risk)
        selected[name] = top
    out["delta_M3_minus_M2"] = out["M3_test_C"] - out["M2_test_C"]
    return out, selected


# ======================================================================
# SEED-42 AUDIT (must pass before the 100 splits)
# ======================================================================
section("SEED-42 AUDIT")
t0 = time.time()

d42, t42 = ordered_split(42)
check(d42 == orig_dev and t42 == orig_test, "seed-42 split IDs and order match locked split")

keep42 = usable_mask(d42)
check(keep42.sum() == 3669, "development-only filter gives 3,669 features")
genes42 = GENES[keep42]

for label, fpath, M in [("M2", M2_SAVED_FILE, expr), ("M3", M3_SAVED_FILE, enesh)]:
    saved = pd.read_csv(fpath, index_col=0)
    saved.index = [norm_pid(x) for x in saved.index]
    saved.columns = [strip_version(x) for x in saved.columns]
    check(set(saved.columns) == set(genes42), f"{label} gene set matches saved matrix")
    check(np.allclose(M.loc[saved.index, saved.columns].to_numpy(float),
                      saved.to_numpy(float), rtol=1e-10, atol=1e-12),
          f"{label} values match saved matrix")

y42 = make_y(survival.loc[d42])
di42 = [ROW[p] for p in d42]
for label, M in [("M2", EXPR_NP), ("M3", ENESH_NP)]:
    Xd = M[:, keep42][di42]
    fast = rank_features(Xd, genes42, y42)
    slow = rank_features_loop(Xd, genes42, y42)
    check(np.allclose(fast["score"].to_numpy(), slow["score"].to_numpy(), atol=1e-12)
          and fast["feature"].head(50).tolist() == slow["feature"].head(50).tolist(),
          f"{label} vectorized ranking identical to original loop")

res42, _ = run_split(42)
print(f"\nSeed-42 fixed-spec M2 test C: {res42['M2_test_C']:.6f} (manuscript {ORIGINAL_M2_C})")
print(f"Seed-42 fixed-spec M3 test C: {res42['M3_test_C']:.6f} (original {ORIGINAL_M3_C_EXACT})")
check(abs(res42["M2_test_C"] - ORIGINAL_M2_C) < 5e-4, "M2 reproduces manuscript internal-test C")
check(abs(res42["M3_test_C"] - ORIGINAL_M3_C_EXACT) < 1e-8, "M3 reproduces original internal-test C")
print(f"Audit time: {(time.time() - t0) / 60:.1f} min")


# ======================================================================
# 100 PARTITIONS
# ======================================================================
section(f"RUNNING {N_SPLITS} PARTITIONS (fixed V4 specifications)")
t0 = time.time()
outputs = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(run_split)(s) for s in range(N_SPLITS))
print(f"Runtime: {(time.time() - t0) / 60:.1f} min")

results = pd.DataFrame([o[0] for o in outputs])
feat_rows = [{"model": m, "split_seed": o[0]["split_seed"], "rank": r, "feature": g}
             for o in outputs for m in ("M2", "M3") for r, g in enumerate(o[1][m], 1)]
features = pd.DataFrame(feat_rows)


# ======================================================================
# SUMMARIES
# ======================================================================
def summarize(v, orig, name):
    v = pd.Series(v, dtype=float)
    return {"Model": name, "N": len(v), "Mean": v.mean(), "SD": v.std(ddof=1),
            "Median": v.median(), "Q1": v.quantile(.25), "Q3": v.quantile(.75),
            "Min": v.min(), "Max": v.max(),
            "P2.5": v.quantile(.025), "P97.5": v.quantile(.975),
            "Original_C": orig,
            "Original_percentile": 100 * np.mean(v <= orig) if orig is not None else np.nan,
            "Prop_gt_0.5": np.mean(v > 0.5)}


summary = pd.DataFrame([
    summarize(results["M2_test_C"], ORIGINAL_M2_C, "M2 expression"),
    summarize(results["M3_test_C"], ORIGINAL_M3_C_EXACT, "M3 eNESH"),
    summarize(results["delta_M3_minus_M2"], ORIGINAL_M3_C_EXACT - ORIGINAL_M2_C, "M3 - M2"),
])
paired = {"Prop_M3_gt_M2": float(np.mean(results["delta_M3_minus_M2"] > 0)),
          "Spearman_M2_M3": float(results[["M2_test_C", "M3_test_C"]].corr("spearman").iloc[0, 1])}

feat_freq = (features.groupby(["model", "feature"]).size()
             .rename("count").reset_index()
             .assign(frequency=lambda d: d["count"] / N_SPLITS)
             .sort_values(["model", "count"], ascending=[True, False]))

section("RESULTS")
pd.set_option("display.width", 200)
print(summary.round(4).to_string(index=False))
print("\nPaired comparison:", {k: round(v, 3) for k, v in paired.items()})
for m in ("M2", "M3"):
    print(f"\nTop 15 {m} features by selection frequency:")
    print(feat_freq[feat_freq["model"] == m].head(15).to_string(index=False))

results.to_csv(OUT / "FIXED_SPEC_100_SPLIT_RESULTS.csv", index=False)
summary.to_csv(OUT / "FIXED_SPEC_100_SPLIT_SUMMARY.csv", index=False)
feat_freq.to_csv(OUT / "FIXED_SPEC_FEATURE_SELECTION_FREQUENCY.csv", index=False)
with open(OUT / "FIXED_SPEC_PAIRED.json", "w") as fh:
    json.dump(paired, fh, indent=2)


# ======================================================================
# FIGURE (one 3-panel PNG)
# ======================================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, col, orig, title in [(ax[0], "M2_test_C", ORIGINAL_M2_C, "M2 expression"),
                            (ax[1], "M3_test_C", ORIGINAL_M3_C_EXACT, "M3 eNESH")]:
    a.hist(results[col], bins=15, color="#7a9cc6", edgecolor="white")
    a.axvline(orig, ls="--", lw=2, color="#c0392b", label=f"Original split C = {orig:.3f}")
    a.axvline(0.5, ls=":", lw=2, color="black", label="C = 0.5")
    a.set_xlabel("Holdout C-index"); a.set_ylabel("Partitions"); a.set_title(title)
    a.legend(fontsize=8)
lo = results[["M2_test_C", "M3_test_C"]].min().min()
hi = results[["M2_test_C", "M3_test_C"]].max().max()
ax[2].scatter(results["M2_test_C"], results["M3_test_C"], alpha=0.7, color="#7a9cc6")
ax[2].plot([lo, hi], [lo, hi], ls="--", color="grey")
ax[2].set_xlabel("M2 C-index"); ax[2].set_ylabel("M3 C-index"); ax[2].set_title("M2 vs M3 per partition")
plt.tight_layout()
plt.savefig(OUT / "FIXED_SPEC_SPLIT_SENSITIVITY.png", dpi=300, bbox_inches="tight")
plt.show()

section("DONE")
print("Outputs in:", OUT)
print("Send back the SEED-42 AUDIT and RESULTS sections.")